## Transformação de dados - pedidos_itens

#### 1.Carregar tabela do bronze

In [1]:
import sys
sys.path.append("/app")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tabela_nome = "pedidos_itens"

spark = create_spark_session("pedidos_itens")
config = load_config()

# ler bronze
df = spark.read.format("delta").load(
    f"data/bronze/{tabela_nome}"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 01:43:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/12 01:43:44 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


#### 2.Executar transformação

In [2]:

# transformação
df_silver = (
    df
    .select("codigo_pedido", "numero_item", "produto_id", "quantidade", "data_entrega", "pedido_id", "id")
    .withColumn("data_entrega", F.to_date("data_entrega", "yyyy-MM-dd"))
)

#### 3.Armazenar dados na camada silver

In [3]:
# salvar silver
path = f"data/silver/{tabela_nome}"

if DeltaTable.isDeltaTable(spark, path):

    delta_table = DeltaTable.forPath(spark, path)

    (
        delta_table.alias("t")
        .merge(
            df_silver.alias("s"),
            """
            t.id = s.id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    df_silver.write.format("delta").save(path)